<a href="https://colab.research.google.com/github/Raedkh317/Extreme-Connectivity-Analysis-Artificial-Intelligence-Prediction-and-Optimal-Portfolio-Strategies/blob/main/quantum_computing_for_financial_portfolio_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q qiskit qiskit-aer qiskit-optimization qiskit-algorithms scipy numpy pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
import pandas as pd
assets = ["BTC", "Gold", "SP500"]
# Expected annual returns (illustrative)
mu = np.array([0.15, 0.07, 0.10])

# Annual volatility assumptions
vol = np.array([0.60, 0.15, 0.18])

# Assumed correlation matrix
corr = np.array([
    [1.00, 0.05, 0.25],
    [0.05, 1.00, 0.10],
    [0.25, 0.10, 1.00]
])

# Annual covariance matrix
cov = np.outer(vol, vol) * corr

print("Expected returns:")
print(pd.Series(mu, index=assets))

print("\nCovariance matrix:")
print(pd.DataFrame(cov, index=assets, columns=assets))

Expected returns:
BTC      0.15
Gold     0.07
SP500    0.10
dtype: float64

Covariance matrix:
          BTC    Gold   SP500
BTC    0.3600  0.0045  0.0270
Gold   0.0045  0.0225  0.0027
SP500  0.0270  0.0027  0.0324


In [ ]:
from scipy.optimize import minimize
import numpy as np

# S'assurer que les variables requises sont définies si la cellule précédente n'a pas été exécutée
try:
    assets
    mu
    cov
except NameError:
    assets = ["BTC", "Gold", "SP500"]
    mu = np.array([0.15, 0.07, 0.10])
    vol = np.array([0.60, 0.15, 0.18])
    corr = np.array([
        [1.00, 0.05, 0.25],
        [0.05, 1.00, 0.10],
        [0.25, 0.10, 1.00]
    ])
    cov = np.outer(vol, vol) * corr

gamma = 0.5
n = len(assets)

def portfolio_objective(w):
    risk = w @ cov @ w
    expected_return = w @ mu
    return risk - gamma * expected_return

constraints = [
    {"type": "eq", "fun": lambda w: np.sum(w) - 1}
]

bounds = [(0, 1)] * n
initial_weights = np.ones(n) / n

result = minimize(
    portfolio_objective,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

weights = result.x

print("Optimal weights:")
for asset, weight in zip(assets, weights):
    print(f"{asset}: {weight:.2%}")

print("Expected annual return:", weights @ mu)
print("Annual volatility:", np.sqrt(weights @ cov @ weights))

Optimal weights:
BTC: 4.35%
Gold: 44.22%
SP500: 51.44%
Expected annual return: 0.08890963743601721
Annual volatility: 0.12751643675632243


In [ ]:
import sys
import subprocess

# S'assurer que qiskit_optimization et qiskit_algorithms sont installés
try:
    from qiskit_optimization import QuadraticProgram
except ModuleNotFoundError:
    print("Installation de qiskit-optimization en cours...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "qiskit-optimization"])
    from qiskit_optimization import QuadraticProgram

try:
    from qiskit_algorithms import QAOA
except ModuleNotFoundError:
    print("Installation de qiskit-algorithms en cours...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "qiskit-algorithms"])
    from qiskit_algorithms import QAOA

from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

# QUBO model
qp = QuadraticProgram()

for asset in assets:
    qp.binary_var(name=asset)

# Objective:
# risk - gamma * expected return
# with portfolio weights = x / 2

linear = {
    assets[i]: cov[i, i] / 4 - gamma * mu[i] / 2
    for i in range(n)
}

quadratic = {
    (assets[i], assets[j]): cov[i, j] / 2
    for i in range(n)
    for j in range(i + 1, n)
}

qp.minimize(
    linear=linear,
    quadratic=quadratic
)

# Choose exactly two assets
qp.linear_constraint(
    linear={asset: 1 for asset in assets},
    sense="==",
    rhs=2,
    name="choose_two"
)

print(qp.prettyprint())

Installation de qiskit-algorithms en cours...
Problem name: 

Minimize
  0.00225*BTC*Gold + 0.0135*BTC*SP500 + 0.00135*Gold*SP500 + 0.0525*BTC
  - 0.011875000000000002*Gold - 0.016900000000000002*SP500

Subject to
  Linear constraints (1)
    BTC + Gold + SP500 == 2  'choose_two'

  Binary variables (3)
    BTC Gold SP500



In [ ]:
sampler = StatevectorSampler(
    default_shots=2048,
    seed=42
)

qaoa = QAOA(
    sampler=sampler,
    optimizer=COBYLA(maxiter=100),
    reps=1
)

quantum_optimizer = MinimumEigenOptimizer(qaoa)

qaoa_result = quantum_optimizer.solve(qp)

print("QAOA solution:")
print(qaoa_result)

x_qaoa = np.rint(qaoa_result.x).astype(int)

print("\nSelected assets:")
for asset, x in zip(assets, x_qaoa):
    if x == 1:
        print(asset)

/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.13/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/

QAOA solution:
fval=-0.027425, BTC=0.0, Gold=1.0, SP500=1.0, status=SUCCESS

Selected assets:
Gold
SP500


In [ ]:
from itertools import combinations

classical_solutions = []

for selected in combinations(range(n), 2):

    x = np.zeros(n)
    x[list(selected)] = 1

    # Equal allocation among the two selected assets
    w = x / 2

    expected_return = w @ mu
    variance = w @ cov @ w
    objective = variance - gamma * expected_return

    classical_solutions.append({
        "Portfolio": " + ".join(
            assets[i] for i in selected
        ),
        "Expected Return": expected_return,
        "Volatility": np.sqrt(variance),
        "Objective": objective,
        "x": x
    })

classical_solutions.sort(
    key=lambda item: item["Objective"]
)

for solution in classical_solutions:
    print(
        solution["Portfolio"],
        "| Return:", round(solution["Expected Return"], 4),
        "| Volatility:", round(solution["Volatility"], 4),
        "| Objective:", round(solution["Objective"], 5)
    )

best_classical = classical_solutions[0]

print("\nBest classical discrete portfolio:")
print(best_classical["Portfolio"])

Gold + SP500 | Return: 0.085 | Volatility: 0.1228 | Objective: -0.02743
BTC + Gold | Return: 0.11 | Volatility: 0.3128 | Objective: 0.04288
BTC + SP500 | Return: 0.125 | Volatility: 0.3341 | Objective: 0.0491

Best classical discrete portfolio:
Gold + SP500


In [ ]:
# Evaluate the QAOA portfolio using the same financial objective
w_qaoa = x_qaoa / 2

qaoa_return = w_qaoa @ mu
qaoa_volatility = np.sqrt(w_qaoa @ cov @ w_qaoa)
qaoa_objective = (
    w_qaoa @ cov @ w_qaoa
    - gamma * (w_qaoa @ mu)
)

print("QAOA selected portfolio:")
print([
    assets[i] for i in range(n)
    if x_qaoa[i] == 1
])

print(f"Expected return: {qaoa_return:.2%}")
print(f"Annual volatility: {qaoa_volatility:.2%}")
print(f"Objective: {qaoa_objective:.6f}")

print("\nExact classical objective:",
      round(best_classical["Objective"], 6))

QAOA selected portfolio:
['Gold', 'SP500']
Expected return: 8.50%
Annual volatility: 12.28%
Objective: -0.027425

Exact classical objective: -0.027425
